# unnoise on Google Colab

**GitHub does not add an “Open in Colab” button to notebook files.** Use the badge in the repo README, paste this link in your browser, or in Colab: *File → Open notebook → GitHub* → `sivaratrisrinivas/unnoise` → `colab/unnoise.ipynb`.

https://colab.research.google.com/github/sivaratrisrinivas/unnoise/blob/main/colab/unnoise.ipynb

---

1. **Runtime → Change runtime type → GPU**
2. Run the cells in order.
3. The last cell starts the server, **embeds the UI with `serve_kernel_port_as_iframe` first** (that registers the port with Colab’s proxy — without it, `*.prod.colab.dev` links often **404**). Then it prints a **proxy URL** for opening in another tab. **Open that tab in the same browser profile where Colab is logged in**; pasting the URL into another browser/device usually gives 404. **Never use `localhost:8000` on your own PC** — that is your machine, not Colab.

If `git clone` fails (private repo), zip this project on your machine, **Upload** it in Colab, unzip, and `%cd` into the folder instead of cloning.

In [ ]:
!pip -q install -U pip
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install diffusers transformers safetensors pillow

In [ ]:
# Replace with your repo if different; or upload a zip and skip this cell.
!git clone https://github.com/sivaratrisrinivas/unnoise.git
%cd unnoise

In [ ]:
import html as html_module
import os
import socket
import subprocess
import sys
import time

from google.colab.output import eval_js
from IPython.display import HTML, display

# Must match where `git clone` put the repo (previous cell uses %cd unnoise).
REPO = "/content/unnoise"
PORT = 8000
assert os.path.isfile(os.path.join(REPO, "app.py")), (
    f"Expected {REPO}/app.py — run the clone + %cd cell first, or set REPO to your folder."
)

LOG_PATH = "/tmp/unnoise_app.log"
env = os.environ.copy()
env["HOST"] = "0.0.0.0"
env["PORT"] = str(PORT)
env["DEVICE"] = "cuda"
env["UI_MAX_FRAMES"] = "all"
# WARMUP_MODEL=1 loads HF weights *before* the HTTP server binds — first run can take
# many minutes. 0 = server binds fast; first "Generate" loads weights.
env["WARMUP_MODEL"] = "0"

with open(LOG_PATH, "w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        [sys.executable, "app.py"],
        cwd=REPO,
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
    )


def wait_for_port(port: int, timeout_s: float = 120) -> bool:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1.0):
                return True
        except OSError:
            time.sleep(1)
            print(".", end="", flush=True)
    return False


print("Starting app.py — logging to", LOG_PATH)
print(f"Waiting for port {PORT} (max ~2 min) …", end="", flush=True)
if not wait_for_port(PORT, timeout_s=120):
    print("\n--- server log (tail) ---")
    print(open(LOG_PATH, "r", encoding="utf-8", errors="replace").read()[-8000:])
    raise RuntimeError(f"Nothing listening on {PORT} — see log above.")

print("\nServer is up on the Colab VM.")

# Prove GET / works from inside the VM (else debug log).
import urllib.error
import urllib.request

try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=10) as resp:
        print("Health check: GET / -> HTTP", resp.status)
except urllib.error.URLError as exc:
    print("\n--- server log (tail) ---")
    print(open(LOG_PATH, "r", encoding="utf-8", errors="replace").read()[-8000:])
    raise RuntimeError(f"GET / failed from inside Colab: {exc}") from exc

from google.colab import output as colab_output

# IMPORTANT: On newer Colab (*.prod.colab.dev), calling proxyPort / new-tab alone can 404
# until the port is registered. serve_kernel_port_as_iframe registers the tunnel.
print("Embedding UI (also registers Colab port forwarding)…")
colab_output.serve_kernel_port_as_iframe(PORT, path="/", height=780)

proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
if not str(proxy_url).startswith("http"):
    raise RuntimeError(f"Unexpected proxyPort return value: {proxy_url!r}")

print("\nProxy URL (new tab — same browser / Google login as this Colab tab):\n", proxy_url)
print(
    "\n*** If this 404s in another browser or device, that’s expected — tunnel is tied to "
    "this Colab session. Use the embedded frame above or open the link from this browser. ***"
)
print("\n*** Do NOT use localhost:8000 on your own PC — that is your computer, not Colab. ***")

safe_href = html_module.escape(proxy_url, quote=True)
display(
    HTML(
        "<p><strong>Optional — same browser as Colab only:</strong></p>"
        f'<p><a href="{safe_href}" target="_blank" rel="noopener" '
        'style="font-size:1.1em">Open unnoise in new tab →</a></p>'
        "<p style=\"color:#666\">Pasting the URL elsewhere often returns HTTP 404.</p>"
    )
)